In [1]:
import sys, json, warnings
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

ROOT = Path.cwd()
while not (ROOT / "outputs").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
OUT = ROOT / "outputs"

clean = pd.read_pickle(OUT / "clean.pkl")
feat  = pd.read_pickle(OUT / "features.pkl")

RESULTS_PATH = OUT / "results.csv"
RESULTS = pd.read_csv(RESULTS_PATH).to_dict("records") if RESULTS_PATH.exists() else []

print("loaded:")
print("  clean :", clean.shape)
print("  feat  :", feat.shape)
print("  prior results:", len(RESULTS), "rows")
print("  python:", sys.executable)

loaded:
  clean : (45657, 16)
  feat  : (45507, 37)
  prior results: 28 rows
  python: c:\Users\hp\AppData\Local\Programs\Python\Python312\python.exe


In [2]:
def log_result(model, tier, protocol, features, encoding,
               y_true, y_pred, n_train=None, notes=""):
    row = dict(model=model, tier=tier, protocol=protocol,
               features=features, encoding=encoding,
               rmse=round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 2),
               mae =round(float(mean_absolute_error(y_true, y_pred)), 2),
               r2  =round(float(r2_score(y_true, y_pred)), 4),
               n_train=n_train, n_test=int(len(y_true)), notes=notes)
    RESULTS.append(row)
    pd.DataFrame(RESULTS).to_csv(RESULTS_PATH, index=False)
    print(f"[{tier:12s}|{encoding:14s}|{features:22s}] {model:28s} "
          f"RMSE={row['rmse']:7.2f} MAE={row['mae']:6.2f} R2={row['r2']:7.4f}")
    return row

In [3]:
CUTOFF = pd.Timestamp("2025-10-15")

feat_reset = feat.reset_index()
tr_all = feat_reset[feat_reset.dt <  CUTOFF].copy()
te_all = feat_reset[feat_reset.dt >= CUTOFF].copy()

print(f"train rows: {len(tr_all)}   test rows: {len(te_all)}")

train rows: 34281   test rows: 11226


In [4]:
BASE_DROP = ["target","date_ist","time_ist","condition_text","description"]

def _clean(df):
    X = df.drop(columns=[c for c in BASE_DROP if c in df.columns])
    return X.select_dtypes(include=[np.number]) if False else X

def enc_onehot(tr, te):
    Xtr = tr.drop(columns=[c for c in BASE_DROP if c in tr.columns]).copy()
    Xte = te.drop(columns=[c for c in BASE_DROP if c in te.columns]).copy()
    Xtr = pd.get_dummies(Xtr, columns=["location"])
    Xte = pd.get_dummies(Xte, columns=["location"])
    Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)
    Xtr = Xtr.select_dtypes(include=[np.number])
    Xte = Xte.select_dtypes(include=[np.number])
    mtr = Xtr.notna().all(axis=1); mte = Xte.notna().all(axis=1)
    return Xtr[mtr], tr.loc[mtr,"target"], Xte[mte], te.loc[mte,"target"]

def enc_ordinal(tr, te):
    Xtr = tr.drop(columns=[c for c in BASE_DROP if c in tr.columns]).copy()
    Xte = te.drop(columns=[c for c in BASE_DROP if c in te.columns]).copy()
    cats = Xtr["location"].astype("category").cat.categories
    Xtr["location"] = pd.Categorical(Xtr["location"], categories=cats).codes
    Xte["location"] = pd.Categorical(Xte["location"], categories=cats).codes
    Xtr = Xtr.select_dtypes(include=[np.number])
    Xte = Xte.select_dtypes(include=[np.number])
    mtr = Xtr.notna().all(axis=1); mte = Xte.notna().all(axis=1)
    return Xtr[mtr], tr.loc[mtr,"target"], Xte[mte], te.loc[mte,"target"]

def enc_target(tr, te, smooth=10):
    Xtr = tr.drop(columns=[c for c in BASE_DROP if c in tr.columns]).copy()
    Xte = te.drop(columns=[c for c in BASE_DROP if c in te.columns]).copy()
    gm = tr["target"].mean()
    stats = tr.groupby("location")["target"].agg(["mean","count"])
    enc = (stats["mean"]*stats["count"] + gm*smooth) / (stats["count"] + smooth)
    Xtr["location"] = Xtr["location"].map(enc).fillna(gm)
    Xte["location"] = Xte["location"].map(enc).fillna(gm)
    Xtr = Xtr.select_dtypes(include=[np.number])
    Xte = Xte.select_dtypes(include=[np.number])
    mtr = Xtr.notna().all(axis=1); mte = Xte.notna().all(axis=1)
    return Xtr[mtr], tr.loc[mtr,"target"], Xte[mte], te.loc[mte,"target"]

def enc_freq(tr, te):
    Xtr = tr.drop(columns=[c for c in BASE_DROP if c in tr.columns]).copy()
    Xte = te.drop(columns=[c for c in BASE_DROP if c in te.columns]).copy()
    freq = tr["location"].value_counts(normalize=True)
    Xtr["location"] = Xtr["location"].map(freq).fillna(0)
    Xte["location"] = Xte["location"].map(freq).fillna(0)
    Xtr = Xtr.select_dtypes(include=[np.number])
    Xte = Xte.select_dtypes(include=[np.number])
    mtr = Xtr.notna().all(axis=1); mte = Xte.notna().all(axis=1)
    return Xtr[mtr], tr.loc[mtr,"target"], Xte[mte], te.loc[mte,"target"]

ENCODERS = {"onehot": enc_onehot, "ordinal": enc_ordinal,
            "target": enc_target, "freq": enc_freq}
print("encoders ready:", list(ENCODERS))

encoders ready: ['onehot', 'ordinal', 'target', 'freq']


In [5]:
from sklearn.ensemble import RandomForestRegressor

for name, enc in ENCODERS.items():
    Xtr, ytr, Xte, yte = enc(tr_all, te_all)
    m = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    m.fit(Xtr, ytr)
    log_result(f"RF[{name}]", "enc_ablation", "A_temporal", "full",
               name, yte, m.predict(Xte), n_train=len(Xtr))

[enc_ablation|onehot        |full                  ] RF[onehot]                   RMSE=   4.88 MAE=  3.05 R2= 0.9760
[enc_ablation|ordinal       |full                  ] RF[ordinal]                  RMSE=   4.88 MAE=  3.06 R2= 0.9760
[enc_ablation|target        |full                  ] RF[target]                   RMSE=   4.83 MAE=  3.01 R2= 0.9765
[enc_ablation|freq          |full                  ] RF[freq]                     RMSE=   4.89 MAE=  3.05 R2= 0.9759


In [6]:
from xgboost import XGBRegressor

def build_features_toggle(df, use_roll=True, use_cyclic=True,
                          use_momentum=True, use_weather=True,
                          use_ewm=True):
    parts = []
    for loc, g in df.groupby(level="location"):
        g = g.copy()
        for L in (1,2,3,6,12,24):
            g[f"aqi_lag{L}"] = g.aqi_index.shift(L)
        if use_roll:
            for W in (6,24):
                g[f"aqi_roll{W}_mean"] = g.aqi_index.shift(1).rolling(W).mean()
                g[f"aqi_roll{W}_max"]  = g.aqi_index.shift(1).rolling(W).max()
                g[f"aqi_roll{W}_std"]  = g.aqi_index.shift(1).rolling(W).std()
        if use_ewm:
            g["aqi_ewm12"] = g.aqi_index.shift(1).ewm(span=12).mean()
        if use_momentum:
            g["aqi_mom3"]    = g.aqi_lag1 - g.aqi_lag3
            g["aqi_ratio24"] = g.aqi_lag1 / (g.get("aqi_roll24_mean", g.aqi_lag1) + 1e-6)
        t = g.index.get_level_values("dt")
        g["hour"]=t.hour; g["month"]=t.month; g["dow"]=t.dayofweek
        g["is_weekend"]=(t.dayofweek>=5).astype(int)
        if use_cyclic:
            g["hour_sin"]=np.sin(2*np.pi*t.hour/24);  g["hour_cos"]=np.cos(2*np.pi*t.hour/24)
            g["month_sin"]=np.sin(2*np.pi*t.month/12); g["month_cos"]=np.cos(2*np.pi*t.month/12)
        if use_weather:
            g["pressure_delta"]=g.pressure_mb.diff()
            g["wind_x_lag1"]   =g.windspeed_kph*g.aqi_lag1

        g["target"] = g.aqi_index.shift(-1)
        g = g.drop(columns=["aqi_index","pm2_5","pm10","co","no2"])
        parts.append(g)
    return pd.concat(parts).dropna(subset=["target","aqi_lag24"])

configs = {
    "lags_only":        dict(use_roll=False, use_cyclic=False, use_momentum=False,
                             use_weather=False, use_ewm=False),
    "lags+roll":        dict(use_roll=True,  use_cyclic=False, use_momentum=False,
                             use_weather=False, use_ewm=True),
    "lags+roll+cyclic": dict(use_roll=True,  use_cyclic=True,  use_momentum=False,
                             use_weather=False, use_ewm=True),
    "+momentum":        dict(use_roll=True,  use_cyclic=True,  use_momentum=True,
                             use_weather=False, use_ewm=True),
    "full":             dict(use_roll=True,  use_cyclic=True,  use_momentum=True,
                             use_weather=True,  use_ewm=True),
}

for tag, kw in configs.items():
    f2 = build_features_toggle(clean, **kw).reset_index()
    tr2 = f2[f2.dt < CUTOFF]; te2 = f2[f2.dt >= CUTOFF]
    Xtr, ytr, Xte, yte = enc_onehot(tr2, te2)
    m = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                     random_state=42, n_jobs=-1)
    m.fit(Xtr, ytr)
    log_result(f"XGB[{tag}]", "feat_ablation", "A_temporal", tag,
               "onehot", yte, m.predict(Xte), n_train=len(Xtr))

[feat_ablation|onehot        |lags_only             ] XGB[lags_only]               RMSE=   5.65 MAE=  3.46 R2= 0.9679
[feat_ablation|onehot        |lags+roll             ] XGB[lags+roll]               RMSE=   4.48 MAE=  2.86 R2= 0.9798
[feat_ablation|onehot        |lags+roll+cyclic      ] XGB[lags+roll+cyclic]        RMSE=   4.14 MAE=  2.52 R2= 0.9827
[feat_ablation|onehot        |+momentum             ] XGB[+momentum]               RMSE=   3.45 MAE=  2.10 R2= 0.9880
[feat_ablation|onehot        |full                  ] XGB[full]                    RMSE=   3.58 MAE=  2.22 R2= 0.9871


In [7]:
from sklearn.ensemble import (ExtraTreesRegressor, AdaBoostRegressor,
                              GradientBoostingRegressor, BaggingRegressor)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import lightgbm as lgb

Xtr, ytr, Xte, yte = enc_onehot(tr_all, te_all)

models = {
    "ExtraTrees":       ExtraTreesRegressor(n_estimators=400, random_state=42, n_jobs=-1),
    "AdaBoost":         AdaBoostRegressor(n_estimators=200, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=200, random_state=42),
    "Bagging":          BaggingRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "LightGBM":         lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05,
                                          n_jobs=-1, verbose=-1),
    "KNN":              make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=15)),
    "SVR-RBF":          make_pipeline(StandardScaler(), SVR(kernel="rbf", C=10, gamma="scale")),
    "MLP":              make_pipeline(StandardScaler(),
                                      MLPRegressor(hidden_layer_sizes=(128,64),
                                                   max_iter=400, random_state=42)),
}

for name, m in models.items():
    m.fit(Xtr, ytr)
    log_result(name, "T2", "A_temporal", "full", "onehot",
               yte, m.predict(Xte), n_train=len(Xtr))

[T2          |onehot        |full                  ] ExtraTrees                   RMSE=   3.58 MAE=  2.29 R2= 0.9871
[T2          |onehot        |full                  ] AdaBoost                     RMSE=  15.85 MAE= 14.74 R2= 0.7470
[T2          |onehot        |full                  ] GradientBoosting             RMSE=   3.49 MAE=  2.10 R2= 0.9877
[T2          |onehot        |full                  ] Bagging                      RMSE=   4.80 MAE=  3.03 R2= 0.9768
[T2          |onehot        |full                  ] LightGBM                     RMSE=   3.55 MAE=  2.21 R2= 0.9873
[T2          |onehot        |full                  ] KNN                          RMSE=  27.29 MAE= 20.93 R2= 0.2502
[T2          |onehot        |full                  ] SVR-RBF                      RMSE=   8.08 MAE=  6.34 R2= 0.9343
[T2          |onehot        |full                  ] MLP                          RMSE=  13.53 MAE= 10.15 R2= 0.8157


In [8]:
from sklearn.ensemble import VotingRegressor, StackingRegressor
from sklearn.linear_model import Ridge, LinearRegression

Xtr, ytr, Xte, yte = enc_onehot(tr_all, te_all)

base = [
    ("lr",  LinearRegression()),
    ("rf",  RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    ("lgb", lgb.LGBMRegressor(n_estimators=300, n_jobs=-1, verbose=-1)),
]

vot = VotingRegressor(estimators=base, n_jobs=-1)
vot.fit(Xtr, ytr)
log_result("Voting(LR+RF+LGBM)", "T3", "A_temporal", "full", "onehot",
           yte, vot.predict(Xte), n_train=len(Xtr))

stk = StackingRegressor(estimators=base, final_estimator=Ridge(), n_jobs=-1)
stk.fit(Xtr, ytr)
log_result("Stacking(→Ridge)", "T3", "A_temporal", "full", "onehot",
           yte, stk.predict(Xte), n_train=len(Xtr))

[T3          |onehot        |full                  ] Voting(LR+RF+LGBM)           RMSE=   3.41 MAE=  2.12 R2= 0.9883
[T3          |onehot        |full                  ] Stacking(→Ridge)             RMSE=   4.87 MAE=  3.08 R2= 0.9761


{'model': 'Stacking(→Ridge)',
 'tier': 'T3',
 'protocol': 'A_temporal',
 'features': 'full',
 'encoding': 'onehot',
 'rmse': 4.87,
 'mae': 3.08,
 'r2': 0.9761,
 'n_train': 34281,
 'n_test': 11226,
 'notes': ''}

In [9]:
tbl = pd.DataFrame(RESULTS).sort_values(["protocol","rmse"]).reset_index(drop=True)
display(tbl)
tbl.to_csv(OUT / "results_all.csv", index=False)

print("\n-- best per tier --")
print(tbl.groupby("tier").first()[["model","rmse","mae","r2"]])

# save best model for later Streamlit demo
import pickle
Xtr, ytr, Xte, yte = enc_onehot(tr_all, te_all)
best_row = tbl.iloc[0]
if "XGB" in best_row.model:
    from xgboost import XGBRegressor
    best_model = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                              random_state=42, n_jobs=-1).fit(Xtr, ytr)
elif "LightGBM" in best_row.model:
    best_model = lgb.LGBMRegressor(n_estimators=400, n_jobs=-1, verbose=-1).fit(Xtr, ytr)
else:
    from sklearn.linear_model import LinearRegression
    best_model = LinearRegression().fit(Xtr, ytr)

pickle.dump({"model": best_model, "columns": list(Xtr.columns),
             "cutoff": str(CUTOFF), "best_model_name": best_row.model},
            open(OUT / "best_model.pkl","wb"))
print(f"\nsaved best model ({best_row.model}) → outputs/best_model.pkl")

,model,tier,protocol,features,encoding,rmse,mae,r2,n_train,n_test,notes
0,Voting(LR+RF+LGBM),T3,A_temporal,full,onehot,3.41,2.12,0.9883,34281,11226,NaN
1,Voting(LR+RF+LGBM),T3,A_temporal,full,onehot,3.41,2.12,0.9883,34281,11226,
2,XGB[+momentum],feat_ablation,A_temporal,+momentum,onehot,3.45,2.10,0.9880,34281,11226,NaN
3,XGB[+momentum],feat_ablation,A_temporal,+momentum,onehot,3.45,2.10,0.9880,34281,11226,
4,GradientBoosting,T2,A_temporal,full,onehot,3.49,2.10,0.9877,34281,11226,NaN
5,GradientBoosting,T2,A_temporal,full,onehot,3.49,2.10,0.9877,34281,11226,
6,XGBoost-tuned,T1,A_temporal,full,onehot_location,3.52,2.14,0.9875,34281,11226,"{'learning_rate': 0.05, 'max_depth': 6, 'n_est..."
7,LightGBM,T2,A_temporal,full,onehot,3.55,2.21,0.9873,34281,11226,NaN
8,LightGBM,T2,A_temporal,full,onehot,3.55,2.21,0.9873,34281,11226,
9,XGBoost,T1,A_temporal,full,onehot_location,3.58,2.22,0.9871,34281,11226,NaN



-- best per tier --
                            model  rmse   mae      r2
tier                                                 
T0                    Persistence  3.90  2.02  0.9847
T1                  XGBoost-tuned  3.52  2.14  0.9875
T2               GradientBoosting  3.49  2.10  0.9877
T3             Voting(LR+RF+LGBM)  3.41  2.12  0.9883
enc_ablation           RF[target]  4.83  3.01  0.9765
feat_ablation      XGB[+momentum]  3.45  2.10  0.9880

saved best model (Voting(LR+RF+LGBM)) → outputs/best_model.pkl


In [10]:
# ---- CLEANUP CELL ----
import pandas as pd
from pathlib import Path

p = Path("outputs/results.csv")
df = pd.read_csv(p)
print("before dedupe:", len(df))

df["encoding"] = df["encoding"].fillna("n/a")
df["notes"]    = df["notes"].fillna("")
df["features"] = df["features"].fillna("n/a")

key = ["model","tier","protocol","features","encoding"]
df = df.drop_duplicates(subset=key, keep="last")
df.to_csv(p, index=False)
print("after dedupe :", len(df))
print("\nper tier:")
print(df.groupby("tier").size())

# reload RESULTS in this kernel too
RESULTS.clear()
RESULTS.extend(df.to_dict("records"))

before dedupe: 47
after dedupe : 26

per tier:
tier
T0               3
T1               4
T2               8
T3               2
enc_ablation     4
feat_ablation    5
dtype: int64


In [11]:

# ---- WEATHER ABLATION CELL ----
from xgboost import XGBRegressor
import pandas as pd, numpy as np
from pathlib import Path

OUT = Path("outputs")
clean = pd.read_pickle(OUT / "clean.pkl")
CUTOFF = pd.Timestamp("2025-10-15")

def build_with(use_weather):
    parts = []
    for loc, g in clean.groupby(level="location"):
        g = g.copy()
        for L in (1,2,3,6,12,24): g[f"aqi_lag{L}"] = g.aqi_index.shift(L)
        for W in (6,24):
            g[f"aqi_roll{W}_mean"]=g.aqi_index.shift(1).rolling(W).mean()
            g[f"aqi_roll{W}_max"] =g.aqi_index.shift(1).rolling(W).max()
            g[f"aqi_roll{W}_std"] =g.aqi_index.shift(1).rolling(W).std()
        g["aqi_ewm12"]=g.aqi_index.shift(1).ewm(span=12).mean()
        g["aqi_mom3"]=g.aqi_lag1-g.aqi_lag3
        g["aqi_ratio24"]=g.aqi_lag1/(g["aqi_roll24_mean"]+1e-6)
        t=g.index.get_level_values("dt")
        g["hour"]=t.hour; g["month"]=t.month; g["dow"]=t.dayofweek
        g["is_weekend"]=(t.dayofweek>=5).astype(int)
        g["hour_sin"]=np.sin(2*np.pi*t.hour/24); g["hour_cos"]=np.cos(2*np.pi*t.hour/24)
        g["month_sin"]=np.sin(2*np.pi*t.month/12); g["month_cos"]=np.cos(2*np.pi*t.month/12)
        if use_weather:
            g["pressure_delta"]=g.pressure_mb.diff()
            g["wind_x_lag1"]=g.windspeed_kph*g.aqi_lag1
        g["target"]=g.aqi_index.shift(-1)
        g = g.drop(columns=["aqi_index","pm2_5","pm10","co","no2"])
        parts.append(g)
    return pd.concat(parts).dropna(subset=["target","aqi_lag24"])

drop = ["target","date_ist","time_ist","condition_text","description"]
res = {}
for tag, use_wx in [("with_weather", True), ("without_weather", False)]:
    f = build_with(use_wx).reset_index()
    tr = f[f.dt < CUTOFF]; te = f[f.dt >= CUTOFF]
    Xtr = pd.get_dummies(tr.drop(columns=[c for c in drop if c in tr.columns]), columns=["location"])
    Xte = pd.get_dummies(te.drop(columns=[c for c in drop if c in te.columns]), columns=["location"])
    Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)
    Xtr = Xtr.select_dtypes(include=[np.number]); Xte = Xte.select_dtypes(include=[np.number])
    m = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=6,
                     random_state=42, n_jobs=-1).fit(Xtr, tr["target"])
    rmse = float(np.sqrt(((m.predict(Xte) - te["target"])**2).mean()))
    res[tag] = round(rmse, 2)

print("weather ablation:")
for k,v in res.items(): print(f"  {k:18s} RMSE = {v}")
print(f"\ndelta (with - without) = {res['with_weather'] - res['without_weather']:+.2f}")

weather ablation:
  with_weather       RMSE = 3.58
  without_weather    RMSE = 3.45

delta (with - without) = +0.13


In [12]:
import pandas as pd
df = pd.read_csv("outputs/results.csv").sort_values(["protocol","rmse"])
print(df[["tier","model","rmse","mae","r2"]].to_string(index=False))
print(f"\nTotal unique models: {len(df)}")

         tier                 model  rmse   mae      r2
           T3    Voting(LR+RF+LGBM)  3.41  2.12  0.9883
feat_ablation        XGB[+momentum]  3.45  2.10  0.9880
           T2      GradientBoosting  3.49  2.10  0.9877
           T1         XGBoost-tuned  3.52  2.14  0.9875
           T2              LightGBM  3.55  2.21  0.9873
           T1               XGBoost  3.58  2.22  0.9871
feat_ablation             XGB[full]  3.58  2.22  0.9871
           T2            ExtraTrees  3.58  2.29  0.9871
           T0           Persistence  3.90  2.02  0.9847
feat_ablation XGB[lags+roll+cyclic]  4.14  2.52  0.9827
           T0      LinearRegression  4.28  2.72  0.9816
feat_ablation        XGB[lags+roll]  4.48  2.86  0.9798
           T2               Bagging  4.80  3.03  0.9768
 enc_ablation            RF[target]  4.83  3.01  0.9765
           T3      Stacking(→Ridge)  4.87  3.08  0.9761
           T1          RandomForest  4.88  3.05  0.9760
 enc_ablation            RF[onehot]  4.88  3.05 

In [1]:
import shutil, pandas as pd, json
from pathlib import Path
OUT = Path("outputs")
shutil.copy(OUT/"results.csv", OUT/"results_v2_all_models.csv")
print("backed up results → results_v2_all_models.csv")

# also save the two ablations as their own CSVs for the paper
df = pd.read_csv(OUT/"results.csv")
df[df.tier=="enc_ablation"].to_csv(OUT/"table_encoding_ablation.csv", index=False)
df[df.tier=="feat_ablation"].to_csv(OUT/"table_feature_ablation.csv", index=False)

evidence = json.loads((OUT/"leak_evidence.json").read_text())
evidence["weather_ablation"] = {"with_weather": 3.58, "without_weather": 3.45, "delta": 0.13,
                                "interpretation": "exogenous weather features hurt"}
(OUT/"leak_evidence.json").write_text(json.dumps(evidence, indent=2))
print("saved ablation tables + evidence")

backed up results → results_v2_all_models.csv
saved ablation tables + evidence
